In [6]:
# Install required libraries in Colab
!pip -q install trafilatura beautifulsoup4 requests spacy
!python -m spacy download en_core_web_sm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 18.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [8]:
import re
import logging
import requests
import spacy
import trafilatura

from bs4 import BeautifulSoup
from collections import Counter
from urllib.parse import urlparse
from google.colab import files


logging.getLogger("trafilatura").setLevel(logging.ERROR)

USER_AGENT = "Mozilla/5.0 (compatible; EducationalSummaryBot/1.0)"


BOILERPLATE_KEYWORDS = [
    "click here",
    "read more",
    "scroll through",
    "come visit",
    "all rights reserved",
    "copyright",
    "privacy policy",
    "terms and conditions",
    "follow us",
    "facebook",
    "twitter",
    "instagram",
    "youtube",
    "login",
    "sign in",
    "subscribe",
    "newsletter",
    "image",
    "menu",
    "home about",
]


NOISY_SECTION_WORDS = [
    "nav",
    "navbar",
    "menu",
    "footer",
    "header",
    "sidebar",
    "carousel",
    "slider",
    "gallery",
    "breadcrumb",
    "social",
    "popup",
    "modal",
    "cookie",
    "advertisement",
    "ads",
]


def normalize_url(url):
    url = url.strip()

    if not url.startswith(("http://", "https://")):
        url = "https://" + url

    parsed = urlparse(url)

    if not parsed.netloc:
        raise ValueError("Invalid URL")

    return url


def scrape_website(url):
    url = normalize_url(url)

    headers = {
        "User-Agent": USER_AGENT,
        "Accept": "text/html,application/xhtml+xml",
        "Accept-Language": "en-US,en;q=0.9",
    }

    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    response.encoding = response.apparent_encoding or response.encoding

    return response.text, response.url


def clean_text(text):
    text = text.replace("\xa0", " ")
    text = re.sub(r"\[\s*\d+\s*\]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def is_useful_block(text):
    text = clean_text(text)
    lower = text.lower()

    if len(text) < 45:
        return False

    if any(word in lower for word in BOILERPLATE_KEYWORDS):
        return False

    words = text.split()

    if len(words) < 7:
        return False

    alpha_chars = sum(ch.isalpha() for ch in text)

    if alpha_chars / max(len(text), 1) < 0.45:
        return False

    # Removes lines that are mostly contact numbers, symbols, or menu fragments.
    if len(set(words)) < max(4, len(words) * 0.35):
        return False

    return True


def fingerprint(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.strip()[:220]


def deduplicate_blocks(blocks):
    seen = set()
    unique = []

    for block in blocks:
        block = clean_text(block)
        fp = fingerprint(block)

        if not fp or fp in seen:
            continue

        seen.add(fp)
        unique.append(block)

    return unique


def remove_noisy_sections(soup):
    # Remove clearly unwanted tags first
    for tag in soup(["script", "style", "noscript", "svg", "canvas", "iframe", "form", "button", "input"]):
        tag.decompose()

    elements_to_remove = []

    # First collect noisy elements
    for element in soup.find_all(True):
        if not hasattr(element, "attrs") or element.attrs is None:
            continue

        element_id = element.get("id") or ""
        element_class = " ".join(element.get("class") or [])
        element_role = element.get("role") or ""

        element_info = f"{element_id} {element_class} {element_role}".lower()

        if any(word in element_info for word in NOISY_SECTION_WORDS):
            elements_to_remove.append(element)

    # Then remove them after the loop
    for element in elements_to_remove:
        try:
            element.decompose()
        except Exception:
            pass

def extract_clean_text(html, url):
    # First try article-style extraction.
    extracted = trafilatura.extract(
        html,
        url=url,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
        output_format="txt"
    )

    blocks = []

    if extracted:
        for line in extracted.splitlines():
            line = clean_text(line)

            if is_useful_block(line):
                blocks.append(line)

    # Fallback for college/company homepages.
    if len(" ".join(blocks)) < 500:
        soup = BeautifulSoup(html, "html.parser")
        remove_noisy_sections(soup)

        main = (
            soup.find("main")
            or soup.find("article")
            or soup.select_one("[role='main']")
            or soup.body
            or soup
        )

        for tag in main.find_all(["h1", "h2", "h3", "p", "li"]):
            line = clean_text(tag.get_text(" ", strip=True))

            if is_useful_block(line):
                blocks.append(line)

    blocks = deduplicate_blocks(blocks)

    return "\n".join(blocks)


def summarize_text(text, nlp, max_sentences=5):
    if not text or len(text.strip()) < 150:
        return "Not enough meaningful text was found on the page."

    # Split by lines first because scraped articles often contain bullet points
    raw_blocks = text.splitlines()

    blocks = []

    for block in raw_blocks:
        block = clean_text(block)

        if not block:
            continue

        # Remove bullet markers
        block = re.sub(r"^[-•*]\s*", "", block).strip()

        if is_useful_block(block):
            blocks.append(block)

    blocks = deduplicate_blocks(blocks)

    if not blocks:
        return "No meaningful text found for summarization."

    # Always keep the first strong introductory block
    intro = blocks[0]

    candidate_sentences = []

    for block in blocks:
        doc = nlp(block)

        for sent in doc.sents:
            sentence = clean_text(sent.text)
            sentence = re.sub(r"^[-•*]\s*", "", sentence).strip()

            if is_useful_block(sentence):
                candidate_sentences.append(sentence)

    candidate_sentences = deduplicate_blocks(candidate_sentences)

    if not candidate_sentences:
        return intro

    keyword_tokens = []

    for sentence in candidate_sentences:
        sent_doc = nlp(sentence)

        for token in sent_doc:
            if (
                not token.is_stop
                and not token.is_punct
                and token.is_alpha
                and len(token.text) > 2
            ):
                keyword_tokens.append(token.lemma_.lower())

    word_freq = Counter(keyword_tokens)

    if not word_freq:
        return " ".join(candidate_sentences[:max_sentences])

    scored = []

    important_topic_words = [
        "architecture",
        "attention",
        "transformer",
        "sequence",
        "context",
        "dependency",
        "encoder",
        "decoder",
        "embedding",
        "neural",
        "language",
        "vision"
    ]

    weak_start_words = [
        "so,",
        "after",
        "both",
        "these",
        "this",
        "each"
    ]

    for index, sentence in enumerate(candidate_sentences):
        lower = sentence.lower()

        lemmas = [
            token.lemma_.lower()
            for token in nlp(sentence)
            if (
                not token.is_stop
                and not token.is_punct
                and token.is_alpha
                and len(token.text) > 2
            )
        ]

        if len(lemmas) < 6:
            continue

        score = sum(word_freq.get(lemma, 0) for lemma in lemmas) / len(lemmas)

        # Prefer sentences that explain the main topic
        if any(word in lower for word in important_topic_words):
            score *= 1.35

        # Penalize sentence fragments that depend on previous context
        if any(lower.startswith(word) for word in weak_start_words):
            score *= 0.55

        # Penalize overly technical small-detail sentences
        if lower.count("token") >= 2:
            score *= 0.75

        # Prefer earlier article sentences
        score *= 1 - (index / max(len(candidate_sentences), 1)) * 0.20

        scored.append((score, index, sentence))

    selected = []

    # Put intro first
    selected.append(intro)

    top = sorted(scored, reverse=True)

    for score, index, sentence in top:
        if sentence == intro:
            continue

        if sentence not in selected:
            selected.append(sentence)

        if len(selected) >= max_sentences:
            break

    return " ".join(selected)

def get_keywords(text, nlp, limit=10):
    doc = nlp(text)

    words = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and token.is_alpha
        and len(token.text) > 2
    ]

    return [word for word, count in Counter(words).most_common(limit)]


def save_summary_to_file(url, cleaned_text, summary, keywords, filename="summary.txt"):
    with open(filename, "w", encoding="utf-8") as file:
        file.write("Website Text Summary\n")
        file.write("====================\n\n")

        file.write(f"Source URL: {url}\n\n")

        file.write("Summary:\n")
        file.write("--------\n")
        file.write(summary)
        file.write("\n\n")

        file.write("Top Keywords:\n")
        file.write("-------------\n")
        file.write(", ".join(keywords))
        file.write("\n\n")

        file.write("Cleaned Text Preview:\n")
        file.write("---------------------\n")
        file.write(cleaned_text[:2500])


url = input("Enter a URL: ")
output_file = "summary.txt"

nlp = spacy.load("en_core_web_sm")

html, final_url = scrape_website(url)
cleaned_text = extract_clean_text(html, final_url)
summary = summarize_text(cleaned_text, nlp, max_sentences=5)
keywords = get_keywords(cleaned_text, nlp)

save_summary_to_file(final_url, cleaned_text, summary, keywords, output_file)

print("Summary saved successfully!\n")
print(summary)

files.download(output_file)

Enter a URL: https://www.geeksforgeeks.org/machine-learning/getting-started-with-transformers/
Summary saved successfully!

Transformer is a neural network architecture used for various machine learning tasks, especially in natural language processing and computer vision. It focuses on understanding relationships within data to process information more effectively. The encoder processes the input sequence into a vector, while the decoder converts this vector back into a sequence. Processes entire sequences at once instead of step by step Transformer architecture uses attention to process an entire sentence at once instead of reading words sequentially. The self attention mechanism allows transformers to determine which words in a sentence are most relevant to each other.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu

In [11]:
# Importing libraries
import re
import textwrap
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [17]:
# Smarter chunking

def load_and_chunk(text, chunk_size=4, overlap=2, min_chunk_chars=120):
    """
    Improvements:
    - Strips bullet markers so '- Foo' doesn't become its own answer
    - Larger window (4 sents, overlap 2) → better context
    - Drops chunks that are too short to be informative
    """
    # Strip bullet markers globally so they don't confuse retrieval
    text = re.sub(r"(?m)^\s*[-•*]\s*", "", text)

    sentences = [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+", text)
        if len(s.strip()) > 30
    ]

    if not sentences:
        raise ValueError("No usable sentences found in cleaned_text.")

    chunks = []
    step = max(1, chunk_size - overlap)

    for i in range(0, len(sentences), step):
        window = sentences[i : i + chunk_size]
        chunk = " ".join(window).strip()
        if len(chunk) >= min_chunk_chars:
            chunks.append(chunk)

    print(f"✅ {len(sentences)} sentences → {len(chunks)} chunks")
    return chunks, sentences


chunks, sentences = load_and_chunk(cleaned_text, chunk_size=4, overlap=2)

# Re-embed because chunks changed
chunk_embeddings = embedder.encode(
    chunks, batch_size=32, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
)

# Rebuild index
index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)
print(f"✅ Re-indexed: {index.ntotal} chunks")

✅ 54 sentences → 27 chunks


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Re-indexed: 27 chunks


In [18]:
# Embed chunks
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # ~80MB, CPU-friendly

print(f"⏳ Encoding {len(chunks)} chunks...")

chunk_embeddings = embedder.encode(
    chunks,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # enables cosine sim via dot product
)

print(f"✅ Embeddings shape: {chunk_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Encoding 27 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings shape: (27, 384)


In [19]:
# Build FAISS index
dim = chunk_embeddings.shape[1]

# IndexFlatIP = exact inner-product = cosine similarity
# (works correctly because embeddings are L2-normalised)
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"✅ FAISS index ready — {index.ntotal} vectors, dim={dim}")

✅ FAISS index ready — 27 vectors, dim=384


In [20]:
# Query-aware answer generation

OVERVIEW_TRIGGERS = [
    "what is", "what's", "what does", "tell me about",
    "describe", "overview", "summary", "about",
    "explain", "main topic", "main idea",
]

def is_overview_query(q: str) -> bool:
    q = q.lower().strip()
    return any(q.startswith(t) or t in q for t in OVERVIEW_TRIGGERS)


def retrieve(query, top_k=5, min_score=0.15):
    """Top_k bumped to 5 so we have more context to work with."""
    q_vec = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True,
    )
    scores, indices = index.search(q_vec, top_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if score >= min_score:
            results.append({
                "chunk_id": int(idx),
                "score":    round(float(score), 4),
                "text":     chunks[idx],
            })
    return results


def build_overview_answer(context_chunks, max_sentences=3):
    """
    For broad questions, combine the top-scoring sentences across
    retrieved chunks into a short paragraph (extractive multi-sentence).
    """
    all_sentences = []
    for item in context_chunks:
        for s in re.split(r"(?<=[.!?])\s+", item["text"]):
            s = s.strip()
            if 40 <= len(s) <= 300:
                all_sentences.append((s, item["score"]))

    # Deduplicate while preserving order
    seen, unique = set(), []
    for s, sc in all_sentences:
        key = s.lower()[:80]
        if key not in seen:
            seen.add(key)
            unique.append((s, sc))

    # Take the top N by chunk score
    top = sorted(unique, key=lambda x: -x[1])[:max_sentences]
    # Re-order by original appearance for readability
    top_set = {s for s, _ in top}
    ordered = [s for s, _ in unique if s in top_set]

    return " ".join(ordered) if ordered else context_chunks[0]["text"]


def extract_specific_answer(query, context_chunks):
    """Single-sentence extractive answer for specific questions."""
    q_vec = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True,
    )

    best_score, best_sentence = -1.0, ""

    for item in context_chunks:
        sents = [
            s.strip()
            for s in re.split(r"(?<=[.!?])\s+", item["text"])
            if len(s.strip()) > 25
        ]
        if not sents:
            continue
        s_vecs = embedder.encode(
            sents, convert_to_numpy=True, normalize_embeddings=True,
        )
        sims = (s_vecs @ q_vec.T).flatten()

        # Blend sentence similarity with chunk-level score
        for i, sim in enumerate(sims):
            combined = 0.7 * sim + 0.3 * item["score"]
            if combined > best_score:
                best_score = combined
                best_sentence = sents[i]

    return best_sentence or context_chunks[0]["text"]


def rag_query(question, top_k=5, show_sources=False):
    print(f"\n{'='*60}")
    print(f"❓  {question}")
    print(f"{'='*60}")

    results = retrieve(question, top_k=top_k)

    if not results:
        print("⚠️  No relevant content found. Try rephrasing.")
        return

    if is_overview_query(question):
        answer = build_overview_answer(results, max_sentences=3)
        mode = "overview"
    else:
        answer = extract_specific_answer(question, results)
        mode = "specific"

    print(f"\n💡 Answer ({mode}):\n   {textwrap.fill(answer, 80, subsequent_indent='   ')}")

    if show_sources:
        print(f"\n📄 Sources ({len(results)} chunks):")
        for r in results:
            print(f"\n  [chunk {r['chunk_id']}]  score={r['score']}")
            print(textwrap.fill(
                r["text"], 80,
                initial_indent="  ", subsequent_indent="  "
            ))

In [21]:
# Interactive Q&A loop

# Quick test with auto-generated questions from your keywords
print("🔍 Auto-testing with top keywords from your scrape...\n")
if keywords:
    rag_query(f"What does the page say about {keywords[0]}?")
if len(keywords) > 1:
    rag_query(f"What is mentioned about {keywords[1]}?")

# Interactive loop — type your own questions
print("\n\n--- 💬 Ask anything about the scraped page (type 'exit' to stop) ---")
while True:
    q = input("\nYour question: ").strip()
    if not q or q.lower() in ("exit", "quit"):
        print("Exiting RAG session.")
        break
    rag_query(q, top_k=3, show_sources=True)

🔍 Auto-testing with top keywords from your scrape...


❓  What does the page say about word?

💡 Answer (overview):
   The token with the highest probability becomes the predicted next word. NLP
   Tasks: Transformers are used for machine translation, text summarization,
   named entity recognition and sentiment analysis. Speech Recognition: They
   process audio signals to convert speech into transcribed text.

❓  What is mentioned about transformer?

💡 Answer (overview):
   Transformer is a neural network architecture used for various machine learning
   tasks, especially in natural language processing and computer vision. It
   focuses on understanding relationships within data to process information
   more effectively. This helps overcome limitations of models like RNNs and
   LSTMs that process data step by step.


--- 💬 Ask anything about the scraped page (type 'exit' to stop) ---

Your question: What are the main functionality of Transformer?

❓  What are the main functionality 